<a href="https://colab.research.google.com/github/NguyenManhCuong1512/AI/blob/main/BaiTap1NhanDangChuSoVietTay.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

In [8]:
# ===== Activation =====
def sigmoid(x):
    return 1/(1+np.exp(-x))

def sigmoid_derivative(x):
    return x*(1-x)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)


In [9]:
# ===== Load data =====
X, y = fetch_openml('mnist_784', version=1, return_X_y=True)

X = X / 255.0
y = y.astype(int)

# One-hot encoding
enc = OneHotEncoder(sparse_output=False)
y = enc.fit_transform(y.to_numpy().reshape(-1,1))

# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [10]:
# ===== Model =====
class NeuralNetwork:
    def __init__(self):
        self.w1 = np.random.randn(784,128) * 0.01
        self.w2 = np.random.randn(128,10) * 0.01

    def forward(self, X):
        self.z1 = np.dot(X, self.w1)
        self.a1 = sigmoid(self.z1)

        self.z2 = np.dot(self.a1, self.w2)
        self.output = softmax(self.z2)
        return self.output

    def backward(self, X, y, lr=0.1):
        m = X.shape[0]

        delta2 = self.output - y
        dW2 = np.dot(self.a1.T, delta2) / m

        delta1 = np.dot(delta2, self.w2.T) * sigmoid_derivative(self.a1)
        dW1 = np.dot(X.T, delta1) / m

        self.w1 -= lr * dW1
        self.w2 -= lr * dW2

    def predict(self, X):
        out = self.forward(X)
        return np.argmax(out, axis=1)

In [11]:
# ===== Train =====
nn = NeuralNetwork()

for epoch in range(10):
    nn.forward(X_train)
    nn.backward(X_train, y_train, lr=0.1)

    loss = -np.mean(y_train * np.log(nn.output + 1e-8))
    print(f"Epoch {epoch}: Loss = {loss:.4f}")


Epoch 0: Loss = 0.2304
Epoch 1: Loss = 0.2302
Epoch 2: Loss = 0.2301
Epoch 3: Loss = 0.2301
Epoch 4: Loss = 0.2300
Epoch 5: Loss = 0.2300
Epoch 6: Loss = 0.2300
Epoch 7: Loss = 0.2300
Epoch 8: Loss = 0.2299
Epoch 9: Loss = 0.2299


In [12]:
# ===== Evaluate =====
pred = nn.predict(X_test)
true = np.argmax(y_test, axis=1)

accuracy = np.mean(pred == true)
print("\nAccuracy:", accuracy)


Accuracy: 0.11114285714285714
